In [1]:
import pandas as pd
import csv
import requests
import xml.etree.ElementTree as ET
from math import cos, asin, sqrt

In [3]:
#import vissim link info file 
csv_file_name = 'C:\\Users\\ets\\Desktop\\RealTwins-Abhilasha\\git_vissim_dev\\ScenarioGenerator\\vissim_link_detailed_info_2_lat_long_modified.xlsx'
df_vissim_ink_info_raw = pd.read_excel(csv_file_name)

In [4]:
print (df_vissim_ink_info_raw)

     Link_No    Link_Name  Ped_Area   Link-or-Connector  FromLinkNo  \
0          1  610-0-Right     False       possibly_link         NaN   
1          2  611-0-Right     False       possibly_link         NaN   
2          3  612-0-Right     False       possibly_link         NaN   
3          4  613-0-Right     False       possibly_link         NaN   
4          5  614-0-Right     False       possibly_link         NaN   
..       ...          ...       ...                 ...         ...   
877    10494          NaN     False  possibly_connector       379.0   
878    10495          NaN     False  possibly_connector       380.0   
879    10496          NaN     False  possibly_connector       381.0   
880    10497          NaN     False  possibly_connector       382.0   
881    10498          NaN     False  possibly_connector       383.0   

     FomLaneNo  FromLinkPos  ToLinkNo  ToLaneNo  ToLinkPos  StartLinkCoord-X  \
0          NaN          NaN       NaN       NaN        NaN        -

In [5]:
links_only_df = df_vissim_ink_info_raw[df_vissim_ink_info_raw['Link-or-Connector'] == 'possibly_link'] 
connectors_only_df = df_vissim_ink_info_raw[df_vissim_ink_info_raw['Link-or-Connector'] == 'possibly_connector'] 

In [6]:
link_list = links_only_df.Link_No.unique().tolist()
connector_list = connectors_only_df.Link_No.unique().tolist()

In [7]:
veh_input_link_list =[]
for i in range(len(link_list)):
    link_id = link_list[i]
    filtered_connector = connectors_only_df[connectors_only_df['ToLinkNo']==link_id]
    if len(filtered_connector.index)== 0:
        print (link_id)
        veh_input_link_list.append(link_id)
        

7
56
58
63
65
66
67
68
77
95
113
121
122
132


In [8]:
print (veh_input_link_list)

[7, 56, 58, 63, 65, 66, 67, 68, 77, 95, 113, 121, 122, 132]


## Entry opendrive road id to entry Vissim link id

In [9]:
def get_opendrive_id_for_sumo_edge(edge_id):
    file_name_sumo_opendrive_ids = 'C:\\Users\\ets\\Desktop\\RealTwins-Abhilasha\\sourcetree_demand_generation_clone\\sumo-opendrive_edge_mapping.xlsx'
    df_sumo_opendrive_ids = pd.read_excel(file_name_sumo_opendrive_ids)
    dictionary_sumo_opendrive_ids =  df_sumo_opendrive_ids.to_dict('records')
    DataList = [p for p in dictionary_sumo_opendrive_ids if (p['sumoID_modified'] == str("'")+edge_id+str("'"))]
    print (DataList[0]['opendriveID'])
    opendriveID = DataList[0]['opendriveID']
    return opendriveID

In [10]:
get_opendrive_id_for_sumo_edge('1481')

686


686

In [11]:
file_name_edge_veh_counts = 'C:\\Users\\ets\\Desktop\\RealTwins-Abhilasha\\sourcetree_demand_generation_clone\\first_edge_veh_counts_v2_filtered_only_relevant.xlsx'
df_edge_veh_counts = pd.read_excel(file_name_edge_veh_counts)
sumo_unique_entry_edge_list = df_edge_veh_counts.route_first_edge.unique().tolist()
print (sumo_unique_entry_edge_list)
    

["'1481'", "'-gneE27'", "'9875'", "'1982'", "'-gneE10'", "'-gneE21'", "'-gneE8'", "'-gneE7'", "'78746'", "'gneE35'", "'-gneE12'"]


In [12]:
opendrive_sumo_id_list = []
for i in range(len(sumo_unique_entry_edge_list)):
    edge_id = sumo_unique_entry_edge_list[i].replace("'","")
    opendrive_id = get_opendrive_id_for_sumo_edge(str(edge_id))
    print (edge_id, opendrive_id)
    small_list = [edge_id, opendrive_id]
    opendrive_sumo_id_list.append(small_list)

686
1481 686
674
-gneE27 674
731
9875 731
704
1982 704
665
-gneE10 665
672
-gneE21 672
677
-gneE8 677
676
-gneE7 676
722
78746 722
741
gneE35 741
667
-gneE12 667


In [13]:
file_name_vissim_lat_long_all_roads = 'C:\\Users\\ets\\Desktop\\RealTwins-Abhilasha\sc_test\\vissim_link_detailed_info_2_lat_long.csv'
df_vissim_lat_long_all_roads = pd.read_csv(file_name_vissim_lat_long_all_roads)
dictionary_vissim_all_roads =  df_vissim_lat_long_all_roads.to_dict('records')
# print (dictionary_vissim_all_roads)

from math import cos, asin, sqrt

def distance(lat1, lon1, lat2, lon2):
    p = 0.017453292519943295
    hav = 0.5 - cos((lat2-lat1)*p)/2 + cos(lat1*p)*cos(lat2*p) * (1-cos((lon2-lon1)*p)) / 2
    return 12742 * asin(sqrt(hav))

def closest(data, v):
    return min(data, key=lambda p: distance(v['lat'],v['lon'],p['Latitude'],p['Longitude']))

In [14]:
# Get closest vissim link to given opendrive edge id 
def get_closest_vissim_link_for_opendrive_road(road_id):
    file_name_opendrive_road_info = 'C:\\Users\\ets\\Desktop\\RealTwins-Abhilasha\\sc_test\\opendrive_road_info_lat_long_inventory.csv'
    df_opendrive_road_info = pd.read_csv(file_name_opendrive_road_info)
    # get lat and long of the opendrive road id
    
    road_lat = df_opendrive_road_info[df_opendrive_road_info['road_id']==road_id]['Latitude']
    road_long = df_opendrive_road_info[df_opendrive_road_info['road_id']==road_id]['Longitude']
    v = {'lat': road_lat , 'lon': road_long}
    vissimDataList = dictionary_vissim_all_roads
    link_no_index = closest(vissimDataList, v)["Link_No"]
    return link_no_index
    

In [15]:
get_closest_vissim_link_for_opendrive_road(666)

10314

In [16]:
opendrive_sumo_vissim_list = []
for j in range(len(opendrive_sumo_id_list)):
    opendrive_id = opendrive_sumo_id_list[j][1]
#     print (opendrive_id)
    vissim_link_no = get_closest_vissim_link_for_opendrive_road(opendrive_id)
#     print (vissim_link_no)
    p = opendrive_sumo_id_list[j]
    print (p)
    q = [vissim_link_no]+p
    print (q)
    opendrive_sumo_vissim_list.append(q)
    
print (opendrive_sumo_vissim_list)
entry_ids_dict = {x[0]: x[1:] for x in opendrive_sumo_vissim_list}
print(entry_ids_dict)
    

['1481', 686]
[77, '1481', 686]
['-gneE27', 674]
[65, '-gneE27', 674]
['9875', 731]
[122, '9875', 731]
['1982', 704]
[95, '1982', 704]
['-gneE10', 665]
[56, '-gneE10', 665]
['-gneE21', 672]
[63, '-gneE21', 672]
['-gneE8', 677]
[68, '-gneE8', 677]
['-gneE7', 676]
[67, '-gneE7', 676]
['78746', 722]
[113, '78746', 722]
['gneE35', 741]
[132, 'gneE35', 741]
['-gneE12', 667]
[58, '-gneE12', 667]
[[77, '1481', 686], [65, '-gneE27', 674], [122, '9875', 731], [95, '1982', 704], [56, '-gneE10', 665], [63, '-gneE21', 672], [68, '-gneE8', 677], [67, '-gneE7', 676], [113, '78746', 722], [132, 'gneE35', 741], [58, '-gneE12', 667]]
{77: ['1481', 686], 65: ['-gneE27', 674], 122: ['9875', 731], 95: ['1982', 704], 56: ['-gneE10', 665], 63: ['-gneE21', 672], 68: ['-gneE8', 677], 67: ['-gneE7', 676], 113: ['78746', 722], 132: ['gneE35', 741], 58: ['-gneE12', 667]}


In [17]:
print (entry_ids_dict[132][1])

741


In [18]:
#function to fetch volume 
def get_volume(time_int_start, vissim_link_id):
    try:
        sumo_edge = str("'")+entry_ids_dict[vissim_link_id][0]+str("'")
        print (sumo_edge)

        file_name_edge_veh_counts = 'C:\\Users\\ets\\Desktop\\RealTwins-Abhilasha\\sourcetree_demand_generation_clone\\first_edge_veh_counts_v2_filtered_only_relevant.xlsx'
        df_edge_veh_counts = pd.read_excel(file_name_edge_veh_counts)
    #     print (df_edge_veh_counts[df_edge_veh_counts['route_first_edge'] == str(sumo_edge)])
        vol = df_edge_veh_counts[(df_edge_veh_counts['route_first_edge'] == str(sumo_edge)) & (df_edge_veh_counts['time_interval_lower'] == time_int_start)]['vehicle_count']
        print (vol)
        return vol.values[0]
    except:
#         print (0)
        return 0

    

In [19]:
file_name_edge_veh_counts = 'C:\\Users\\ets\\Desktop\\RealTwins-Abhilasha\\sourcetree_demand_generation_clone\\first_edge_veh_counts_v2_filtered_only_relevant.xlsx'
df_edge_veh_counts = pd.read_excel(file_name_edge_veh_counts)
# dictionary_edge_veh_counts =  df_edge_veh_counts.to_dict('records')
# print (dictionary_edge_veh_counts[0])

In [20]:
dictionary_edge_veh_counts = df_edge_veh_counts.set_index('index').T.to_dict('list')
# df_edge_veh_counts.set_index('index').to_dict()
# print (dictionary_edge_veh_counts)
print (dictionary_edge_veh_counts["'1481'-0"][3])


1


In [21]:
#function to fetch volume 
def get_volume_dict(time_int_start, vissim_link_id):
    try:
        sumo_edge = str("'")+entry_ids_dict[vissim_link_id][0]+str("'")
#         print (sumo_edge)
        
        index_key = str(sumo_edge)+"-"+str(time_int_start)
#         print (index_key)
        vol = dictionary_edge_veh_counts[index_key][3]
#         print (vol)
        return vol
    except:
#         print (0)
        return 0

In [22]:
get_volume_dict(54000, 56)

3

In [23]:
get_volume(54000,56)

'-gneE10'
8580    3
Name: vehicle_count, dtype: int64


3

In [33]:
from __future__ import print_function
import os
# COM-Server
import win32com.client as com

In [34]:
Vissim = com.gencache.EnsureDispatch("Vissim.Vissim") #
Path_to_Vissim_Network = 'C:\\Users\\ets\\Desktop\\RealTwins-Abhilasha\\sourcetree_clone\\vissim\\'

In [35]:
## Load a Vissim Network:
Filename               = os.path.join(Path_to_Vissim_Network, 'chatt3.inpx')
flag_read_additionally = False # you can read network(elements) additionally, in this case set "flag_read_additionally" to true
Vissim.LoadNet(Filename, flag_read_additionally)

## Load a Layout:
Filename = os.path.join(Path_to_Vissim_Network, 'chatt3.layx')
Vissim.LoadLayout(Filename)

In [36]:
# Set simulation parameters
# Choose Random Seed
Random_Seed = 42
Vissim.Simulation.SetAttValue('RandSeed', Random_Seed)

# Set simulation time
simulation_duration = 10800 # simulation second [s]
Vissim.Simulation.SetAttValue('SimPeriod', simulation_duration)

# Set maximum speed:
Vissim.Simulation.SetAttValue('UseMaxSimSpeed', True)

In [37]:

# volume aggregation level in seconds
vol_aggregation_level = 60 

# number of time interval bins that needs to be created
no_of_intervals = int(simulation_duration/vol_aggregation_level)

#
for timeInt in range(2, (no_of_intervals+1)):
        # Add timeinterval y accessing vissim timeinterval for vehicle input parameter
        Vissim.Net.TimeIntervalSets.ItemByKey(1).TimeInts.AddTimeInterval(timeInt)
        # Set start  time for the time interval created in aove command
        TimeIntTag = Vissim.Net.TimeIntervalSets.ItemByKey(1).TimeInts.ItemByKey(timeInt)
        TimeIntTag.SetAttValue('Start',vol_aggregation_level*(timeInt-1))



In [38]:
#create vehilce inputs for each entry link
vissim_veh_input_keys_list = []
for i in range(len(veh_input_link_list)):
    vissim_link_object = Vissim.Net.Links.ItemByKey(veh_input_link_list[i])
    VehInput = Vissim.Net.VehicleInputs.AddVehicleInput(0, vissim_link_object)
    print (VehInput.AttValue('No'))
    vissim_veh_input_keys_list.append(VehInput.AttValue('No'))


1
2
3
4
5
6
7
8
9
10
11
12
13
14


In [ ]:
get_volume(54000,veh_input_link_list[1])

In [39]:
start_time_of_day = 15*3600 

vissim_link_zero_entries = [7, 66, 121]

for j in range(len(veh_input_link_list)):

        VI_number = vissim_veh_input_keys_list[j]

        for t in range(1, no_of_intervals+1):
            #get volume from the table for the timeinterval and entry link
            if veh_input_link_list[j] in vissim_link_zero_entries:
                new_volume = 0
            else:
                new_volume = get_volume_dict(start_time_of_day+((t-1)*60),veh_input_link_list[j])
                print (j, t, new_volume)

            #assign volume to the vehicle input time interval
            Vissim.Net.VehicleInputs.ItemByKey(VI_number).SetAttValue('Volume('+str(t)+')', new_volume*60)


            #assign volume type to the vehicle input time interval
            Vissim.Net.VehicleInputs.ItemByKey(VI_number).SetAttValue('VolType('+str(t)+')', 1)
    #         Vissim.Net.VehicleInputs.ItemByKey(VI_number).SetAttValue('Cont('+str(t)+')', False)
    


1 1 3
1 2 3
1 3 3
1 4 3
1 5 3
1 6 3
1 7 3
1 8 3
1 9 3
1 10 3
1 11 3
1 12 3
1 13 2
1 14 3
1 15 3
1 16 3
1 17 3
1 18 3
1 19 3
1 20 3
1 21 3
1 22 3
1 23 3
1 24 3
1 25 2
1 26 3
1 27 3
1 28 3
1 29 3
1 30 3
1 31 3
1 32 3
1 33 3
1 34 3
1 35 3
1 36 3
1 37 2
1 38 3
1 39 3
1 40 3
1 41 3
1 42 3
1 43 3
1 44 3
1 45 3
1 46 3
1 47 3
1 48 3
1 49 2
1 50 3
1 51 3
1 52 3
1 53 3
1 54 3
1 55 3
1 56 3
1 57 3
1 58 3
1 59 3
1 60 2
1 61 2
1 62 2
1 63 1
1 64 2
1 65 1
1 66 2
1 67 2
1 68 1
1 69 2
1 70 1
1 71 2
1 72 2
1 73 1
1 74 2
1 75 1
1 76 2
1 77 2
1 78 1
1 79 2
1 80 1
1 81 2
1 82 2
1 83 1
1 84 2
1 85 1
1 86 2
1 87 2
1 88 1
1 89 2
1 90 1
1 91 2
1 92 2
1 93 1
1 94 2
1 95 1
1 96 2
1 97 2
1 98 1
1 99 2
1 100 1
1 101 2
1 102 2
1 103 1
1 104 2
1 105 1
1 106 2
1 107 2
1 108 1
1 109 2
1 110 1
1 111 2
1 112 2
1 113 1
1 114 2
1 115 1
1 116 2
1 117 2
1 118 1
1 119 2
1 120 1
1 121 3
1 122 2
1 123 3
1 124 2
1 125 2
1 126 3
1 127 2
1 128 3
1 129 2
1 130 2
1 131 3
1 132 2
1 133 2
1 134 3
1 135 2
1 136 3
1 137 2
1 138 2
1 13

8 17 1
8 18 1
8 19 1
8 20 1
8 21 0
8 22 1
8 23 1
8 24 1
8 25 1
8 26 1
8 27 1
8 28 1
8 29 1
8 30 1
8 31 1
8 32 1
8 33 1
8 34 1
8 35 1
8 36 1
8 37 1
8 38 1
8 39 1
8 40 1
8 41 0
8 42 1
8 43 1
8 44 1
8 45 1
8 46 1
8 47 1
8 48 1
8 49 1
8 50 1
8 51 1
8 52 1
8 53 1
8 54 1
8 55 1
8 56 1
8 57 1
8 58 1
8 59 1
8 60 0
8 61 1
8 62 1
8 63 1
8 64 1
8 65 1
8 66 1
8 67 1
8 68 1
8 69 1
8 70 1
8 71 1
8 72 1
8 73 1
8 74 1
8 75 1
8 76 1
8 77 1
8 78 1
8 79 1
8 80 1
8 81 1
8 82 1
8 83 1
8 84 1
8 85 1
8 86 1
8 87 1
8 88 1
8 89 1
8 90 1
8 91 1
8 92 1
8 93 1
8 94 1
8 95 1
8 96 1
8 97 1
8 98 1
8 99 1
8 100 1
8 101 1
8 102 1
8 103 1
8 104 1
8 105 1
8 106 1
8 107 1
8 108 1
8 109 1
8 110 1
8 111 1
8 112 1
8 113 1
8 114 1
8 115 1
8 116 1
8 117 1
8 118 1
8 119 1
8 120 0
8 121 1
8 122 1
8 123 0
8 124 1
8 125 1
8 126 0
8 127 1
8 128 0
8 129 1
8 130 1
8 131 0
8 132 1
8 133 1
8 134 0
8 135 1
8 136 0
8 137 1
8 138 1
8 139 0
8 140 1
8 141 0
8 142 1
8 143 1
8 144 0
8 145 1
8 146 1
8 147 0
8 148 1
8 149 0
8 150 1
8 151 1
8 1

In [40]:
#Save file
Filename = os.path.join('C:\\Users\\ets\\Desktop\\RealTwins-Abhilasha\\sourcetree_demand_generation_clone\\vissim_model_demand_population\\', 'chatt3_vols.inpx')
Vissim.SaveNetAs(Filename)
Filename = os.path.join('C:\\Users\\ets\\Desktop\\RealTwins-Abhilasha\\sourcetree_demand_generation_clone\\vissim_model_demand_population\\', 'chatt3_vols.layx')
Vissim.SaveLayout(Filename)